# Produce yes/no predictions from two hours of sensor history

This example uses a historical turbine and timestamp to demonstrate inference. It uses the validation-selected saved model for each horizon, with its frozen threshold. No future alarm or sensor values are inputs; no fitting takes place.


In [ ]:
from pathlib import Path
import sys
import json
import pandas as pd
from IPython.display import display, Image, Markdown

REPO = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / 'configs' / 'short_horizon_alarm.json').is_file()), None)
if REPO is None:
    raise FileNotFoundError('Open this notebook from inside the Git repository.')
sys.path.insert(0, str(REPO / 'src'))
from wt_predmaint.short_horizon import load_config
CFG = load_config(REPO / 'configs' / 'short_horizon_alarm.json')
print('Repository:', REPO)


In [ ]:
REPORT_ROOT = REPO / CFG['report_root']
RUN_ID = None  # Paste a folder name to review a specific run.
if RUN_ID is None:
    completed = [p for p in REPORT_ROOT.iterdir() if (p / 'status.json').is_file()
                 and json.loads((p / 'status.json').read_text()).get('status') in ['complete', 'frozen_before_test']]
    if not completed:
        raise FileNotFoundError('No completed run yet. Run notebook 01 first.')
    OUT = sorted(completed)[-1]
else:
    OUT = REPORT_ROOT / RUN_ID
print('Reviewing:', OUT)


In [ ]:
import joblib
from wt_predmaint.short_horizon import predict_history
TURBINE_ID = '2304511'
FORECAST_TIME = pd.Timestamp('2025-02-01 12:00')  # Existing historical example; change as needed.
selected = json.loads((OUT / 'frozen_selection.json').read_text())
sample_bundle = joblib.load(OUT / f"10min_{selected['10']['model']}.joblib")
columns = [*sample_bundle['sensors'], 'sensorObserved']
source = REPO / CFG['data_root'] / str(FORECAST_TIME.year) / f'Station_{TURBINE_ID}' / 'data.parquet'
# Timestamp filters prevent opening future readings for this forecast.
history = pd.read_parquet(source, columns=columns, filters=[
    ('TimeStamp', '>=', FORECAST_TIME - pd.Timedelta(hours=2)),
    ('TimeStamp', '<', FORECAST_TIME),
])
results = []
for horizon in [10, 20, 30]:
    bundle = joblib.load(OUT / f"{horizon}min_{selected[str(horizon)]['model']}.joblib")
    results.append(predict_history(bundle, history, FORECAST_TIME))
display(pd.DataFrame(results))


A forecast is refused if any of the 12 required source intervals is missing. `model_score` is not a calibrated probability. Three separately trained models need not output monotonic scores or decisions across horizons, although the true binary labels are nested.
